# 01.3 Autograd / 自动微分

`Autograd` 是 `PyTorch` 的核心能力之一。  
`Autograd` is one of the core capabilities of `PyTorch`.

如果你把这一节真正弄明白，后面的 `loss.backward()` 就不再只是“照着写”。  
If you really understand this notebook, `loss.backward()` will stop being something you merely copy.

重点概念 / Key concepts:

- 计算图 / computational graph
- `requires_grad`
- `backward()`
- 梯度 / gradients
- 梯度累积 / gradient accumulation
- `no_grad()` 与 `detach()`

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 理解什么是 `requires_grad=True` / Understand what `requires_grad=True` means.
2. 看懂简单计算图 / Read a simple computational graph.
3. 用 `backward()` 求梯度 / Use `backward()` to compute gradients.
4. 理解梯度为什么会累积 / Understand why gradients accumulate.
5. 正确使用 `torch.no_grad()` / Use `torch.no_grad()` correctly.
6. 理解 `detach()` 的作用 / Understand what `detach()` does.

In [ ]:
import torch

## 1. `requires_grad` 与计算图 / `requires_grad` and the Computational Graph

当一个张量设置了 `requires_grad=True`，`PyTorch` 会开始追踪它参与的运算。  
When a tensor has `requires_grad=True`, `PyTorch` starts tracking operations involving it.

这些运算连接起来，就形成了计算图 / computational graph。  
These connected operations form the computational graph.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

print("x =", x)
print("y =", y)
print("x.requires_grad =", x.requires_grad)
print("y.requires_grad =", y.requires_grad)
print("y.grad_fn =", y.grad_fn)

这里可以先从直觉理解：  
A good first intuition is:

- `x` 是需要求导的输入 / `x` is an input we want gradients for
- `y` 是由 `x` 经过一系列运算得到的结果 / `y` is produced from `x` through a sequence of operations
- `grad_fn` 表示 `y` 背后有可追踪的计算历史 / `grad_fn` indicates tracked computation history

## 2. `backward()` 求梯度 / Using `backward()` to Compute Gradients

如果输出是标量 / scalar，直接调用 `backward()` 即可。  
If the output is a scalar, you can call `backward()` directly.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1
y.backward()

print("x.grad =", x.grad)

手算验证 / Manual check:

- `y = x^2 + 3x + 1`
- `dy/dx = 2x + 3`
- 当 `x=2` 时，梯度是 `7`

所以 `x.grad == 7`。  
So `x.grad == 7`.

In [ ]:
# 练习 1 / Exercise 1
# 令 y = 4x^2 - x
# Let y = 4x^2 - x
#
# 1. 令 x=3, requires_grad=True
# 2. 用 backward() 求梯度
# 3. 打印 x.grad

# x =
# y =
# y.backward()
# print(x.grad)

In [ ]:
# 练习 1 参考答案 / Exercise 1 Reference Solution

x = torch.tensor(3.0, requires_grad=True)
y = 4 * x ** 2 - x
y.backward()
print(x.grad)

# 手算 / manually: dy/dx = 8x - 1, at x=3 => 23

## 3. 非标量输出 / Non-Scalar Outputs

如果输出不是标量，`backward()` 需要额外提供梯度权重 / gradient argument。  
If the output is not a scalar, `backward()` needs an additional gradient argument.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

y.backward(gradient=torch.ones_like(y))
print("x.grad =", x.grad)

这里传入 `torch.ones_like(y)`，相当于把每个输出项都等权相加。  
Passing `torch.ones_like(y)` is equivalent to weighting each output term equally.

在实际训练里，`loss` 通常已经是标量，所以更常见的是直接：  
In practice, the training `loss` is usually already a scalar, so the common pattern is simply:

- `loss.backward()`

## 4. 梯度累积 / Gradient Accumulation

这是一个非常关键、非常常见、也非常容易踩坑的点。  
This is a very important, very common, and very easy-to-miss point.

在 `PyTorch` 中，梯度默认会累积到 `.grad` 里。  
In `PyTorch`, gradients accumulate in `.grad` by default.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("第一次 backward 后 / after first backward:", x.grad)

y2 = 3 * x
y2.backward()
print("第二次 backward 后 / after second backward:", x.grad)

为什么会这样 / Why does this happen?

- 第一次：`d(x^2)/dx = 2x = 4`
- 第二次：`d(3x)/dx = 3`
- 累积结果：`4 + 3 = 7`

这就是训练循环里经常要写 `optimizer.zero_grad()` 的原因之一。  
This is one reason why training loops often call `optimizer.zero_grad()`.

In [ ]:
# 练习 2 / Exercise 2
# 复现实验：
# Reproduce the experiment:
# 1. x=1, requires_grad=True
# 2. 先对 y=x^3 backward
# 3. 再对 z=2x backward
# 4. 观察 x.grad 的累积结果

# x =
# y =
# y.backward()
# z =
# z.backward()
# print(x.grad)

In [ ]:
# 练习 2 参考答案 / Exercise 2 Reference Solution

x = torch.tensor(1.0, requires_grad=True)
y = x ** 3
y.backward()
z = 2 * x
z.backward()
print(x.grad)

# dy/dx = 3, dz/dx = 2, total = 5

## 5. 手动清零梯度 / Manually Zeroing Gradients

在优化参数前，通常要先把旧梯度清零。  
Before updating parameters, you usually clear old gradients first.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print("清零前 / before zeroing:", x.grad)

x.grad.zero_()
print("清零后 / after zeroing:", x.grad)

z = 3 * x
z.backward()
print("再次 backward 后 / after backward again:", x.grad)

## 6. `torch.no_grad()` / `torch.no_grad()`

有些场景你不想让 `PyTorch` 跟踪梯度，比如：  
Sometimes you do not want `PyTorch` to track gradients, for example:

- 推理 / inference
- 验证 / validation
- 单纯做数值查看 / plain numeric inspection

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x * 5

print("y =", y)
print("y.requires_grad =", y.requires_grad)

`no_grad()` 的主要价值 / Main value of `no_grad()`:

- 节省显存和计算图开销 / saves graph overhead and memory
- 避免不必要的梯度追踪 / avoids unnecessary gradient tracking

## 7. `detach()` / `detach()`

`detach()` 会返回一个新的张量视图，它共享数据，但不再参与当前计算图。  
`detach()` returns a new tensor view that shares data but no longer participates in the current graph.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = 2 * x
z = y.detach()

print("y.requires_grad =", y.requires_grad)
print("z.requires_grad =", z.requires_grad)

y_sum = y.sum()
y_sum.backward()
print("x.grad =", x.grad)

直觉上可以理解为：  
Intuitively, you can think of it as:

- `y` 还连接在图里 / `y` is still connected to the graph
- `z` 从图里“剪下来”了 / `z` has been cut away from the graph

In [ ]:
# 练习 3 / Exercise 3
# 判断下面哪些张量会追踪梯度。
# Decide which tensors below will track gradients.
#
# x = torch.tensor(2.0, requires_grad=True)
# a = x * 2
# with torch.no_grad():
#     b = x * 3
# c = a.detach()
#
# print(a.requires_grad)
# print(b.requires_grad)
# print(c.requires_grad)

In [ ]:
# 练习 3 参考答案 / Exercise 3 Reference Solution

x = torch.tensor(2.0, requires_grad=True)
a = x * 2
with torch.no_grad():
    b = x * 3
c = a.detach()

print(a.requires_grad)
print(b.requires_grad)
print(c.requires_grad)

## 8. 一个最小训练感知例子 / A Tiny Training-Flavored Example

下面这个例子还不是真正的训练循环，但它已经很接近了。  
The example below is not yet a full training loop, but it is already very close.

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
x = torch.tensor(2.0)
target = torch.tensor(4.0)

pred = w * x
loss = (pred - target) ** 2
loss.backward()

print("pred =", pred.item())
print("loss =", loss.item())
print("w.grad =", w.grad.item())

这里已经出现了训练最核心的链条：  
The core training chain already appears here:

- 参数 / parameter: `w`
- 前向计算 / forward pass: `pred = w * x`
- 损失 / loss: `(pred - target)^2`
- 反向传播 / backward pass: `loss.backward()`

下一节学训练循环时，这条链会变成完整版本。  
When you learn the training loop later, this chain will become the full version.

## 9. 小结 / Summary

你现在应该能回答 / You should now be able to answer:

1. `requires_grad=True` 到底意味着什么？ / What exactly does `requires_grad=True` mean?
2. 为什么标量输出可以直接 `backward()`？ / Why can scalar outputs call `backward()` directly?
3. 为什么 `.grad` 会累积？ / Why does `.grad` accumulate?
4. `no_grad()` 和 `detach()` 的区别是什么？ / What is the difference between `no_grad()` and `detach()`?
5. 为什么训练中常常先清零梯度？ / Why do we usually clear gradients before training steps?

下一步建议 / Suggested next step:

- 接下来可以继续做 `Dataset` / `DataLoader`，或者直接进入 `nn.Module` 和训练循环 / Next you can move on to `Dataset` / `DataLoader`, or directly into `nn.Module` and the training loop.